# Stage 2 — Language-Grounded Functional Segmentation

Training-free pipeline: scan + natural-language task -> 2D mask on the functional element.

**Run order after any disconnect: Cell 1 (SETUP), then Cell 2 (RESUME). Nothing else.**

---

### Measured configuration (do not change without re-measuring)

| choice | value | evidence |
|---|---|---|
| scorer | `proto - proto*bg` | 45.3% vs 38.9% bank (n=120, 3 seeds) |
| background subtraction | on | worth **19.5 points** |
| centering | **off** | costs 8 points despite +0.38 on a separability proxy |
| layer | 1 | tied set {1,2,16}; centered features near-flat across layers |
| box selection | GD rank-0 | 58.3% vs 55.0% for best reranker (n=58) |
| multi-box top-k | 2 | IoU 0.186 vs 0.157 (k=1), 0.181 (k=3), 0.154 (k=5) |
| SAM prompt | on box crop | full-frame gave whole-cabinet masks: coverage 1.0, IoU 0.005 |
| SAM prompt | **point only** | box hint REJECTED on CUDA: 0.101 vs 0.175 (n=80). See below |

### SAM box-prompt: MEASURED NEGATIVE, rejected (was provisionally adopted)

Point-only SAM prompting was letting all 3 candidate masks come back oversized for
small hardware on a large flat panel, so the post-hoc `SAM_AREA_MULT` penalty had
nothing smaller to fall back to. The proposed fix passed the high-similarity
("hot") region's bounding box to SAM as an extra scale hint (`_hot_box`,
`SAM_BOX_PROMPT`). An informal MPS n=30 A/B suggested it doubled mean IoU
(0.029 -> 0.062) and the header previously warned those absolute numbers must be
re-measured on GPU before being quoted.

**Re-measured on the T4 at n=80, and it does not transfer -- it actively hurts:**

| condition | box prompt ON | box prompt OFF | reference |
|---|---|---|---|
| full (oracle label) | 0.101 | **0.175** | 0.175 |
| ceiling (GT box) | 0.143 | **0.251** | 0.251 |

Coverage rises (0.541 vs 0.271) while IoU falls -- the signature of masks that are
too large. Giving SAM an explicit box makes it segment *the box*, not the element
inside it. `SAM_BOX_PROMPT` is therefore **False** in `config.py`, and the earlier
claim that `SAM_AREA_MULT` had become redundant is withdrawn -- that claim assumed
the box prompt was on.

### Peak-coordinate bug in `run()` -- fixed

`pipeline.run()` scaled the correspondence peak's y-coordinate by `(X1-Y0)`, an
x-coordinate minus a y-coordinate, instead of the box height `(Y1-Y0)`. For a box
at x in [1000,1200], y in [100,300] the correct scale is 200 but it computed 1100
-- a ~5.5x vertical stretch that threw the predicted point well below the box and
often off-image.

`evalutils.evaluate`'s own `oracle_box` path carries a separate, correct copy of
the same formula, which is why the ceiling condition still reproduced while the
full pipeline collapsed (in-GT 8.8% vs 32%). An isolated backbone test confirmed
DINOv3 was never at fault: correspondence 45.3% +/- 0.8%, matching the reference
exactly. With both fixes applied, every headline number reproduces to three
decimals.

### Known-good numbers

- retrieval: label@1 67.0%, label@5 89.0%, family@5 96.0% (n=200 cross-scene)
- correspondence: 45.3% peak-in-GT at crop scale
- end-to-end 2D: IoU(closed) 0.157, ceiling (GT box) 0.235

### Open lead, not yet adopted: box reranking with GD confidence

Locally (n=40, single seed): peak-only box selection degrades as `topk` widens (65.0%
at k=2, 52.5% at k=4/k=8 -- confirms your existing k=2 choice). Combining GD's own
detection confidence with the correspondence peak (z-scored, blended per-item) breaks
that degradation: 57.5% at k=2 (worse), but 65.0% at k=4 and **67.5% at k=8** --
recovering past the k=2 peak-only baseline while searching 4x deeper toward the 96.7%
reachable ceiling. This is a real, reproducible local signal but not yet a confirmed
win against your n=60 deployed number -- see the new cell in Section 6 to validate it
at proper scale before adopting.

### Not included here
3D lifting and the official AP/mIoU harness live in a separate notebook.

## 1. SETUP

Run once per session. Handles the Drive-FUSE import bug by copying the package to local disk.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import sys, os, subprocess, shutil

ROOT = '/content/drive/MyDrive/scenefun3d'   # <- adjust if your Drive layout differs
assert os.path.isdir(ROOT), f'not found: {ROOT}'

# Google Drive's FUSE mount is not reliably visible to Python's import finder:
# os.listdir() sees the files but importlib.util.find_spec() returns None.
# Working from a local copy avoids this entirely.
LOCAL = '/content/sf3d'
os.makedirs(f'{ROOT}/sf3d', exist_ok=True)
open(f'{ROOT}/sf3d/__init__.py', 'a').close()
if os.path.isdir(LOCAL): shutil.rmtree(LOCAL)
shutil.copytree(f'{ROOT}/sf3d', LOCAL)
if '/content' not in sys.path: sys.path.insert(0, '/content')

if not os.path.isdir('/content/scenefun3d'):
    subprocess.run(['git','clone','--depth','1',
                    'https://github.com/SceneFun3D/scenefun3d.git'], cwd='/content')
if '/content/scenefun3d' not in sys.path: sys.path.insert(0, '/content/scenefun3d')

try:
    import plyfile, open3d
except ImportError:
    subprocess.run(['pip','install','-q','plyfile','open3d','open_clip_torch','gradio'])

import torch
torch.set_grad_enabled(False)      # this notebook never trains
print('setup ok | cuda:', torch.cuda.is_available())

In [ ]:
import os
from getpass import getpass

# Prefer environment variables for non-interactive runs; fall back to interactive prompt.
auth_token = os.environ.get("NGROK_AUTHTOKEN") or os.environ.get("NGROK_AUTH_TOKEN")
if not auth_token:
    auth_token = getpass("Enter your ngrok auth token: ")
    if not auth_token:
        raise RuntimeError("NGROK auth token is required. Set NGROK_AUTHTOKEN or NGROK_AUTH_TOKEN in the environment.")
# export for tools that read the env
os.environ["NGROK_AUTHTOKEN"] = auth_token

# Install helpers and validate the token by briefly creating an ngrok tunnel via pyngrok.
# This gives a clearer error message if the token is invalid or ngrok cannot start.
%pip install colab_ssh pyngrok --quiet

from colab_ssh import launch_ssh
from pyngrok import ngrok, conf

# Configure pyngrok with the token
conf.get_default().auth_token = auth_token

try:
    # Start a short-lived http tunnel on an arbitrary port to validate the token and ngrok binary
    t = ngrok.connect(8050, "http")
    print('ngrok public url (validation):', t.public_url)
    ngrok.disconnect(t.public_url)
    ngrok.kill()
except Exception as e:
    # Surface a helpful diagnostic rather than the low-level ConnectionRefused traceback
    print('Warning: ngrok validation failed — the token may be invalid or ngrok failed to start.')
    print('Error:', repr(e))
    print('Check your token at https://dashboard.ngrok.com/get-started/your-authtoken and ensure outbound network access is allowed.')
    # Continue to attempt launch_ssh; colab_ssh will raise if it cannot create a tunnel.

# Finally, start the SSH tunnel (colab_ssh itself will manage ngrok/ssh); publish=True requests a public tunnel
try:
    launch_ssh(auth_token, "12345", publish=True)
except Exception as e:
    print('launch_ssh failed:', repr(e))
    raise


## 2. Package modules

Written to local disk, synced to Drive. Edit here, never in a bare cell — rebinding functions across cells is how the earlier notebook ended up with two incompatible `score_box` definitions.

In [ ]:
%%writefile /content/sf3d/config.py
"""Single source of truth. Every tunable lives here and nowhere else."""
import os

ROOT      = '/content/drive/MyDrive/scenefun3d'
DATA      = f'{ROOT}/data'
FRAMES    = f'{ROOT}/frames'
DB_PATH   = f'{ROOT}/exemplar_db.pkl'
CKPT      = f'{ROOT}/dinov3_vitl14_pretrain.pth'
HUB_ID    = 'facebook/dinov3-vitl16-pretrain-lvd1689m'
CLIP_CACHE= f'{ROOT}/clip_embs.npz'

# ---- measured; see notebook header for evidence ----
LAYER       = 1        # tied set {1,2,16}
CENTER      = False    # costs 8 pts of peak-in-GT
BANK_SHORT  = 518
QUERY_SHORT = 518      # deliberately equal: the 518/1036 mismatch was an accident
K_EX        = 3
TOPQ        = 0.10
BOX_PAD     = 0.25     # the only padding; hit tests use pad=0
GD_TOPK     = 8
GD_CONF     = 0.25
MULTIBOX_K  = 2
MAX_AREA_FRAC = 0.02   # masks larger than this are parent objects, not elements
SAM_AREA_MULT = 15.0   # NOT redundant -- that claim assumed SAM_BOX_PROMPT=True, now rejected
SAM_BOX_PROMPT = False # MEASURED NEGATIVE on CUDA: n=80 full IoU 0.101 (on) vs 0.175 (off),
                       # ceiling 0.143 vs 0.251. The MPS n=30 signal did not transfer. See header.

MEDIAN_AREA = None     # filled by pipeline.init_stats(db); see notebook

PARENT_HINT = {
  'hook_pull' : 'cabinet. drawer. door. fridge. oven.',
  'pinch_pull': 'drawer. cabinet. handle.',
  'hook_turn' : 'door. window. lever.',
  'rotate'    : 'knob. dial. valve. faucet.',
  'key_press' : 'switch. button. keypad.',
  'tip_push'  : 'button. switch. panel.',
  'foot_push' : 'pedal. bin. floor.',
  'unplug'    : 'socket. outlet. plug.',
  'plug_in'   : 'socket. outlet. plug.',
}
DEFAULT_HINT = 'handle. knob. switch. button.'

FAMILY = {'hook_pull':'pull','pinch_pull':'pull','unplug':'pull','plug_in':'pull',
          'rotate':'turn','hook_turn':'turn',
          'key_press':'press','tip_push':'press','foot_push':'press'}

In [ ]:
%%writefile /content/sf3d/pipeline.py
"""Backbone, dense features, prototype scoring, Grounding DINO, SAM."""
import os, re, numpy as np, cv2, torch
from . import config as C

_M = {}   # model singletons

# ------------------------------------------------------------------ backbone
def dino():
    if 'dino' in _M: return _M['dino']
    from transformers import AutoConfig, AutoModel
    sd = torch.load(C.CKPT, map_location='cpu', weights_only=False)
    sd = sd.get('model', sd.get('state_dict', sd))
    assert not any('pos_embed' in k for k in sd), 'learned pos_embed -> this is DINOv2'
    pw = next(k for k in sd if k.endswith(('patch_embeddings.weight',
                                           'patch_embed.proj.weight')))
    dim, patch = int(sd[pw].shape[0]), int(sd[pw].shape[-1])
    nl = 1 + max(int(m.group(1)) for k in sd if (m := re.search(r'layers?\.([^.]+)\.', k)))
    reg = next((k for k in sd if 'register' in k), None)
    nprefix = 1 + (int(sd[reg].shape[1]) if reg else 0)

    cfg = AutoConfig.from_pretrained(C.HUB_ID)
    assert (cfg.hidden_size, cfg.patch_size, cfg.num_hidden_layers) == (dim, patch, nl)
    m = AutoModel.from_pretrained(C.HUB_ID, config=cfg); tgt = set(m.state_dict())
    cand, name = min((({f(k): v for k, v in sd.items()}, n) for n, f in {
        'as-is'       : lambda k: k,
        'strip model.': lambda k: re.sub(r'^model\.', '', k),
        'strip 1'     : lambda k: k.split('.', 1)[1] if '.' in k else k}.items()),
        key=lambda c: len(tgt - set(c[0])))
    # strict=False without checking `missing` is how you end up running a
    # randomly-initialised model that produces plausible numbers.
    assert not (tgt - set(cand)), f'remap "{name}" missing {len(tgt-set(cand))} keys'
    m.load_state_dict(cand, strict=False)
    m = m.to('cuda').eval()

    blocks = None
    for n, mod in m.named_modules():
        if isinstance(mod, torch.nn.ModuleList) and len(mod) == nl:
            assert blocks is None, 'ambiguous block list'
            blocks = mod
    assert blocks is not None
    _M['dino'] = dict(model=m, blocks=blocks, dim=dim, patch=patch,
                      n_layers=nl, n_prefix=nprefix)
    print(f'DINOv3 dim={dim} patch={patch} blocks={nl} prefix={nprefix} via "{name}"')
    return _M['dino']

MEAN = np.array([0.485, 0.456, 0.406])
STD  = np.array([0.229, 0.224, 0.225])

class _Stop(Exception): pass

def patch_feats(img, layer=None, short=None):
    """(gh, gw, D) unit-norm patch grid.

    Uses an early-exit forward hook rather than output_hidden_states=True: at
    short=1036 that materialised 25 x ~5600 x 1024 floats (~575 MB) per call
    to use one slice of it, which is where the OOMs came from.
    """
    d = dino(); layer = layer or C.LAYER; short = short or C.QUERY_SHORT
    P = d['patch']; H, W = img.shape[:2]; s = short / min(H, W)
    gh, gw = max(round(H*s/P), 1), max(round(W*s/P), 1)
    im = (cv2.resize(img, (gw*P, gh*P), interpolation=cv2.INTER_CUBIC)
            .astype(np.float32)/255. - MEAN) / STD
    t = torch.from_numpy(im).permute(2,0,1)[None].float().cuda()
    cap = {}
    def hook(mod, i, o):
        cap['x'] = o[0] if isinstance(o, (tuple, list)) else o
        raise _Stop
    h = d['blocks'][d['n_layers']-layer].register_forward_hook(hook)
    try: d['model'](pixel_values=t)
    except _Stop: pass
    finally: h.remove()
    n = gh*gw; tok = cap['x']
    # take the LAST h*w tokens: robust to however many prefix tokens exist
    assert tok.shape[1] == n + d['n_prefix'], f'{tok.shape[1]} != {n}+{d["n_prefix"]}'
    f = tok[:, -n:, :].reshape(gh, gw, -1).float().cpu().detach().numpy()
    del tok, t
    return f / (np.linalg.norm(f, axis=-1, keepdims=True) + 1e-8)

def patchify(mask, gh, gw):
    return cv2.resize(mask.astype(np.uint8), (gw, gh),
                      interpolation=cv2.INTER_NEAREST).astype(bool)

def _erode1(m):
    """Boundary patches mix hardware with background and carry the residual
    reprojection offset, so drop one patch of border."""
    if m.sum() <= 6: return m
    e = cv2.erode(m.astype(np.uint8), np.ones((3,3), np.uint8)).astype(bool)
    return e if e.sum() >= 2 else m

# ------------------------------------------------------- prototype scoring
def build_proto(entries, layer=None, short=None, bg_samples=128):
    """Mean-pooled foreground prototype + background prototype.

    Measured best of four scorers (45.3% vs 38.9% for a top-q bank, n=120).
    The bank's multimodality advantage does not appear at k_ex=3, plausibly
    because three same-label exemplars rarely span multiple modes.
    """
    FG, BG = [], []
    rng = np.random.default_rng(0)
    for e in entries:
        for v in e.get('views', [e])[:3]:
            f = patch_feats(v['crop'], layer, short or C.BANK_SHORT)
            m = patchify(v['mask'], *f.shape[:2])
            fg, bg = f[_erode1(m)], f[~m]
            if len(fg): FG.append(fg)
            if len(bg):
                k = min(bg_samples, len(bg))
                BG.append(bg[rng.choice(len(bg), k, replace=False)])
    if not FG: return None
    u = lambda A: A / (np.linalg.norm(A) + 1e-8)
    return dict(fg=u(np.vstack(FG).mean(0)),
                bg=u(np.vstack(BG).mean(0)) if BG else None,
                layer=layer or C.LAYER, short=short or C.BANK_SHORT)

def score_image(img, proto):
    """Similarity map. Background subtraction is worth ~19.5 points and is the
    single most important component in the pipeline."""
    f = patch_feats(img, proto['layer'], proto['short'])
    f2 = f.reshape(-1, f.shape[-1])
    s = f2 @ proto['fg']
    if proto['bg'] is not None: s = s - f2 @ proto['bg']
    return s.reshape(f.shape[:2]), f.shape[:2]

# ------------------------------------------------------------ Grounding DINO
def gdino():
    if 'gd' in _M: return _M['gd']
    from transformers import AutoProcessor, AutoModelForZeroShotObjectDetection
    mid = 'IDEA-Research/grounding-dino-tiny'
    _M['gd'] = (AutoProcessor.from_pretrained(mid),
                 AutoModelForZeroShotObjectDetection.from_pretrained(mid).cuda().eval())
    return _M['gd']

def gd_boxes(img, text, topk=None):
    from PIL import Image
    proc, mod = gdino(); topk = topk or C.GD_TOPK
    inp = proc(images=Image.fromarray(img), text=text.lower().strip(),
               return_tensors='pt').to('cuda')
    out = mod(**inp)
    r = proc.post_process_grounded_object_detection(
        out, inp.input_ids, threshold=0.15, text_threshold=0.15,
        target_sizes=[img.shape[:2]])[0]
    b, s = r['boxes'].cpu().numpy(), r['scores'].cpu().numpy()
    o = np.argsort(-s)[:topk]
    return b[o], s[o]

# ------------------------------------------------------------------- SAM
def sam():
    if 'sam' in _M: return _M['sam']
    from transformers import SamProcessor, SamModel
    mid = 'facebook/sam-vit-base'
    _M['sam'] = (SamProcessor.from_pretrained(mid),
                 SamModel.from_pretrained(mid).cuda().eval())
    return _M['sam']

def _hot_box(hot, pad_frac=0.15):
    """Bounding box of the high-similarity region, padded a bit. Gives SAM a
    scale hint so it doesn't have to guess between 'the handle' and 'the whole
    cabinet' from a bare point -- see BOX PROMPT note below."""
    ys, xs = np.where(hot)
    if len(xs) == 0: return None
    x0, y0, x1, y1 = xs.min(), ys.min(), xs.max()+1, ys.max()+1
    px, py = pad_frac*(x1-x0), pad_frac*(y1-y0)
    H, W = hot.shape
    return [float(max(0, x0-px)), float(max(0, y0-py)),
            float(min(W, x1+px)), float(min(H, y1+py))]

def sam_mask(img, pt, smap=None, region=None, area_prior=None):
    """SAM prompted on the BOX CROP, not the full frame.

    On a 1920x1440 frame the whole-cabinet mask is the natural segmentation:
    measured failures had coverage 1.0 with IoU 0.005 (correct peak, ~200x too
    large). Cropping first, weighting purity above cover, and rejecting masks
    far larger than a typical annotation doubled the GT-box ceiling (0.117 ->
    0.235).

    BOX PROMPT: point-only prompting still let SAM propose all-oversized mask
    triples for small hardware on a large flat panel -- measured cases where
    every one of the 3 candidates was 30-250x the median GT area, so the area
    penalty below had nothing smaller to fall back to (it penalises all three
    equally and changes nothing). Passing the high-similarity region's own
    bounding box as an additional prompt gives SAM an explicit scale hint.
    Verified n=30 local (MPS): mean IoU 0.029 -> 0.062, good rate 3% -> 20%.
    Toggle with config.SAM_BOX_PROMPT for an A/B check.
    """
    from PIL import Image
    proc, mod = sam()
    if region is not None:
        X0, Y0, X1, Y1 = region
        sub, spt = img[Y0:Y1, X0:X1], (pt[0]-X0, pt[1]-Y0)
    else:
        X0 = Y0 = 0; sub, spt = img, pt

    hot = None
    if smap is not None:
        h = smap[Y0:Y1, X0:X1] if region is not None else smap
        hot = h >= np.percentile(h, 92)
    box_prompt = _hot_box(hot) if (hot is not None and C.SAM_BOX_PROMPT) else None

    kw = dict(input_points=[[[float(spt[0]), float(spt[1])]]])
    if box_prompt is not None:
        kw['input_boxes'] = [[box_prompt]]
    inp = proc(Image.fromarray(sub), return_tensors='pt', **kw).to('cuda')
    out = mod(**inp)
    masks = proc.image_processor.post_process_masks(
        out.pred_masks.cpu(), inp['original_sizes'].cpu(),
        inp['reshaped_input_sizes'].cpu())[0][0].numpy()
    iou = out.iou_scores.cpu().numpy().ravel()

    best, bs = 0, -1e9
    for i, m in enumerate(masks):
        if m.sum() < 20: continue
        s = 0.3 * iou[i]
        if hot is not None:
            cov = (m & hot).sum() / max(hot.sum(), 1)
            pur = (m & hot).sum() / max(m.sum(), 1)
            # combine so that cover 1.0 / purity 0.2 cannot tie 0.6 / 0.6
            s += 2.0*pur + 0.5*cov
        if area_prior and m.sum() > C.SAM_AREA_MULT * area_prior:
            s -= 3.0
        if s > bs: bs, best = s, i
    full = np.zeros(img.shape[:2], bool)
    if region is not None: full[Y0:Y1, X0:X1] = masks[best]
    else: full = masks[best]
    return full, masks, iou

# ------------------------------------------------------------- full 2D run
def run(img, proto, text, topk=None, area_prior=None):
    """Multi-box: score correspondence inside each of the top-K boxes, keep the
    box holding the globally strongest match. Parameter-free -- the peak IS the
    criterion. Distinct from the reranker that failed (55.0% vs 58.3% null),
    which scored boxes on region statistics BEFORE localising within them."""
    topk = topk or C.MULTIBOX_K
    area_prior = area_prior if area_prior is not None else C.MEDIAN_AREA
    boxes, confs = gd_boxes(img, text)
    if len(boxes) == 0: return None
    H, W = img.shape[:2]
    best = None
    for b, cf in zip(boxes[:topk], confs[:topk]):
        x0, y0, x1, y1 = b
        pw, ph = (x1-x0)*C.BOX_PAD, (y1-y0)*C.BOX_PAD
        X0, Y0 = int(max(0, x0-pw)), int(max(0, y0-ph))
        X1, Y1 = int(min(W, x1+pw)), int(min(H, y1+ph))
        if X1-X0 < 16 or Y1-Y0 < 16: continue
        smap, (gh, gw) = score_image(img[Y0:Y1, X0:X1], proto)
        peak = float(smap.max())
        if best is None or peak > best['peak']:
            pi, pj = np.unravel_index(int(np.argmax(smap)), smap.shape)
            best = dict(peak=peak, box=b, conf=float(cf), region=(X0,Y0,X1,Y1),
                        smap=smap,
                        point=(X0+(pj+.5)*(X1-X0)/gw, Y0+(pi+.5)*(Y1-Y0)/gh))
    if best is None: return None
    X0, Y0, X1, Y1 = best['region']
    up = cv2.resize(best['smap'], (X1-X0, Y1-Y0), interpolation=cv2.INTER_CUBIC)
    full = np.full((H, W), float(best['smap'].min()), np.float32)
    full[Y0:Y1, X0:X1] = up
    mask, cands, iou = sam_mask(img, best['point'], full,
                                region=best['region'], area_prior=area_prior)
    return dict(box=best['box'], conf=best['conf'], region=best['region'],
                point=best['point'], mask=mask, smap=full, peak=best['peak'],
                sam_cands=cands, sam_iou=iou)

def init_stats(db):
    """Median annotation area -- a dataset constant, not a per-item oracle."""
    C.MEDIAN_AREA = float(np.median([e['mask'].sum() for e in db.values()]))
    return C.MEDIAN_AREA


In [ ]:
%%writefile /content/sf3d/retrieval.py
"""CLIP hybrid retrieval. Embeddings cached to Drive; re-encoding the whole
exemplar DB per query is otherwise the pipeline bottleneck."""
import os, numpy as np, torch
from . import config as C

_R = {}

def _clip():
    if 'm' not in _R:
        import open_clip
        m, _, pp = open_clip.create_model_and_transforms('ViT-L-14', pretrained='openai')
        _R['m'], _R['pp'] = m.cuda().eval(), pp
        _R['tok'] = open_clip.get_tokenizer('ViT-L-14')
    return _R['m'], _R['pp'], _R['tok']

def _unit(a, ax=-1):
    return a / (np.linalg.norm(a, axis=ax, keepdims=True) + 1e-8)

def build_index(db, force=False):
    if not force and os.path.exists(C.CLIP_CACHE):
        z = np.load(C.CLIP_CACHE, allow_pickle=True)
        _R['keys'] = list(z['keys']); _R['I'] = z['I']
        _R['T'] = z['T']; _R['has_t'] = z['has_t']
        print(f'loaded CLIP cache: {len(_R["keys"])} exemplars')
        return
    from PIL import Image
    m, pp, tok = _clip()
    keys = list(db.keys()); I, T, has_t = [], [], []
    for i, k in enumerate(keys):
        e = db[k]
        with torch.no_grad():
            im = pp(Image.fromarray(e['crop'])).unsqueeze(0).cuda()
            I.append(m.encode_image(im).float().cpu().numpy()[0])
            d = (e.get('desc') or [''])[0]
            if d:
                T.append(m.encode_text(tok([d]).cuda()).float().cpu().numpy()[0])
                has_t.append(True)
            else:
                T.append(np.zeros_like(I[-1])); has_t.append(False)
        if (i+1) % 100 == 0: print(f'  {i+1}/{len(keys)}')
    _R['keys'] = keys; _R['I'] = _unit(np.stack(I))
    _R['T'] = _unit(np.stack(T)); _R['has_t'] = np.array(has_t)
    np.savez_compressed(C.CLIP_CACHE, keys=np.array(keys, object),
                        I=_R['I'], T=_R['T'], has_t=_R['has_t'])
    print(f'encoded and cached {len(keys)} exemplars')

def encode_query(text):
    m, _, tok = _clip()
    with torch.no_grad():
        q = m.encode_text(tok([text]).cuda()).float().cpu().numpy()[0]
    return _unit(q)

def retrieve(query_text, db, k=5, alpha=0.5, max_per_scene=2, exclude_visit=None):
    """s(q,e) = a*<t(q), i(e)> + (1-a)*<t(q), tbar(e)>

    Each modality is centred by its OWN gallery mean and both sides are
    renormalised. Without this, CLIP hubness makes every query return the same
    five results. Exemplars with no description fall back to the image score
    rather than being structurally excluded by a zero text vector.
    """
    keys, I, T, has_t = _R['keys'], _R['I'], _R['T'], _R['has_t']
    q = encode_query(query_text)

    Im = I.mean(0)
    img_s = _unit(I - Im) @ _unit(q - Im)
    txt_s = img_s.copy()
    if has_t.any():
        Tm = T[has_t].mean(0)
        txt_s[has_t] = _unit(T[has_t] - Tm) @ _unit(q - Tm)

    score = alpha*img_s + (1-alpha)*txt_s
    out, seen = [], {}
    for i in np.argsort(-score):
        kk = keys[i]; vis = db[kk]['visit']
        if exclude_visit and vis == exclude_visit: continue
        if seen.get(vis, 0) >= max_per_scene: continue
        out.append(kk); seen[vis] = seen.get(vis, 0) + 1
        if len(out) >= k: break
    return out

def guess_label(db, ex_keys):
    labs = [db[k]['label'] for k in ex_keys]
    return max(set(labs), key=labs.count) if labs else None

In [ ]:
%%writefile /content/sf3d/evalutils.py
"""Metrics and evaluation loops. Kept out of pipeline.py so the pipeline stays
importable without any evaluation assumptions."""
import numpy as np, cv2, os
from . import config as C, pipeline as P, retrieval as RT

def closed_gt(mask, k=21):
    """SceneFun3D GT is a sparse point set: projected and dilated it fills a
    median of 6.1% of its own bounding box, so raw IoU structurally penalises
    correct segmentations. Report against a morphologically closed GT."""
    return cv2.morphologyEx(mask.astype(np.uint8), cv2.MORPH_CLOSE,
                            np.ones((k, k), np.uint8)).astype(bool)

def gt_full(e, H, W):
    x0, y0, x1, y1 = e['box']
    g = np.zeros((H, W), bool)
    sub = e['mask'][:y1-y0, :x1-x0]
    g[y0:y0+sub.shape[0], x0:x0+sub.shape[1]] = sub
    return g

def pool_oracle(db, key, k_ex=None):
    """Same label, other scenes. NOTE: uses the ground-truth label."""
    t = db[key]
    return [db[k] for k, e in db.items()
            if e['visit'] != t['visit'] and e['label'] == t['label']][:(k_ex or C.K_EX)]

def pool_retrieved(db, key, k_ex=5):
    """Retrieval-driven: no GT label anywhere in the prediction path."""
    e = db[key]
    desc = (e.get('desc') or [''])[0]
    if not desc: return [], None
    ks = RT.retrieve(desc, db, k=k_ex, exclude_visit=e['visit'])
    return [db[k] for k in ks], RT.guess_label(db, ks)

def evaluate(db, keys, n=80, seed=0, oracle_box=False, oracle_label=True,
             oracle_hint=True, topk=None, verbose=False):
    """Returns per-item records. Three complementary metrics per handoff 5.1:
    IoU against closed GT, GT coverage, and peak distance in pixels."""
    rng = np.random.default_rng(seed)
    sel = rng.choice(keys, min(n, len(keys)), replace=False)
    R = []
    for k in sel:
        e = db[k]
        fp = f"{C.FRAMES}/{e['frame_file']}"
        if not os.path.exists(fp): continue

        if oracle_label:
            pool, guess = pool_oracle(db, k), e['label']
        else:
            pool, guess = pool_retrieved(db, k)
        if not pool: continue
        proto = P.build_proto(pool)
        if proto is None: continue

        img = cv2.cvtColor(cv2.imread(fp), cv2.COLOR_BGR2RGB)
        H, W = img.shape[:2]
        g = gt_full(e, H, W)
        if g.sum() < 10: continue
        gc = closed_gt(g); gpts = np.argwhere(g)

        hint = C.PARENT_HINT.get(guess, C.DEFAULT_HINT) if oracle_hint else C.DEFAULT_HINT
        text = f"{(e.get('desc') or [''])[0]}. {hint}"

        if oracle_box:
            x0, y0, x1, y1 = e['box']
            pw, ph = (x1-x0)*.4, (y1-y0)*.4
            X0, Y0 = int(max(0, x0-pw)), int(max(0, y0-ph))
            X1, Y1 = int(min(W, x1+pw)), int(min(H, y1+ph))
            smap, (gh, gw) = P.score_image(img[Y0:Y1, X0:X1], proto)
            pi, pj = np.unravel_index(int(np.argmax(smap)), smap.shape)
            pt = (X0+(pj+.5)*(X1-X0)/gw, Y0+(pi+.5)*(Y1-Y0)/gh)
            up = cv2.resize(smap, (X1-X0, Y1-Y0), interpolation=cv2.INTER_CUBIC)
            full = np.full((H, W), float(smap.min()), np.float32)
            full[Y0:Y1, X0:X1] = up
            mask, _, _ = P.sam_mask(img, pt, full, region=(X0,Y0,X1,Y1),
                                    area_prior=C.MEDIAN_AREA)
            res = dict(mask=mask, point=pt, peak=float(smap.max()), conf=1.0)
        else:
            res = P.run(img, proto, text, topk=topk)
            if res is None: continue

        m = res['mask']
        py, px = int(np.clip(res['point'][1], 0, H-1)), int(np.clip(res['point'][0], 0, W-1))
        R.append(dict(key=k, label=e['label'], guess=guess,
                      iou=(m & gc).sum() / max((m | gc).sum(), 1),
                      coverage=(m & g).sum() / g.sum(),
                      peak_px=float(np.min(np.linalg.norm(
                          gpts - np.array([py, px]), axis=1))),
                      in_gt=bool(gc[py, px]),
                      mask_px=int(m.sum()), peak=res.get('peak', np.nan)))
        if verbose: print(f"  {R[-1]['iou']:.3f}  {R[-1]['coverage']:.2f}")
    return R

def summarise(R, name=''):
    if not R:
        print(f'{name}: no results'); return {}
    g = lambda f: np.array([r[f] for r in R])
    s = dict(n=len(R), iou=float(g('iou').mean()), iou_med=float(np.median(g('iou'))),
             coverage=float(g('coverage').mean()),
             peak15=float(np.mean(g('peak_px') < 15)),
             in_gt=float(g('in_gt').mean()),
             peak_med=float(np.median(g('peak_px'))),
             zeros=float(np.mean(g('iou') < 0.02)),
             good=float(np.mean(g('iou') > 0.15)))
    print(f"{name:<26} n={s['n']:>3}  IoU {s['iou']:.3f}  cov {s['coverage']:.3f}  "
          f"peak<15px {s['peak15']:.0%}  in-GT {s['in_gt']:.0%}  "
          f"zeros {s['zeros']:.0%}  good {s['good']:.0%}")
    return s

def distribution(R, name=''):
    """The mean is misleading here: two binary gates (box hit, correspondence
    hit) multiply into a bimodal outcome, not a cluster around the mean."""
    v = np.array([r['iou'] for r in R])
    print(f'\n{name}  n={len(v)}  mean {v.mean():.3f}  median {np.median(v):.3f}')
    bins = [0, .02, .05, .10, .15, .20, .30, .50, 1.01]
    hist, _ = np.histogram(v, bins=bins)
    for lo, hi, c in zip(bins[:-1], bins[1:], hist):
        bar = '#' * int(c / max(hist.max(), 1) * 40)
        print(f'  [{lo:.2f},{hi:.2f})  {c:>3}  {bar}')

In [ ]:
import shutil, importlib, sys

def sync():
    """Push local package edits back to Drive so they survive a disconnect."""
    shutil.copytree(LOCAL, f'{ROOT}/sf3d', dirs_exist_ok=True)
    print('synced to Drive')

for m in [k for k in list(sys.modules) if k.startswith('sf3d')]:
    del sys.modules[m]
from sf3d import config as C, pipeline as P, retrieval as RT, evalutils as EV
sync()
print('modules imported')

## 3. RESUME

Everything below depends on this. After a disconnect: Cell 1, then this.

In [ ]:
import pickle, glob, numpy as np, cv2, os, importlib
for m in (C, P, RT, EV): importlib.reload(m)

DB = pickle.load(open(C.DB_PATH, 'rb'))
KEYS = [k for k, e in DB.items()
        if e.get('frame_file') and os.path.exists(f"{C.FRAMES}/{e['frame_file']}")]
pairs = {}
for e in DB.values(): pairs.setdefault(str(e['visit']), str(e['video']))

P.init_stats(DB)
RT.build_index(DB)

LABELS = sorted({e['label'] for e in DB.values()})
assert 'exclude' not in LABELS, 'exclude class leaked into the DB'
print(f'{len(DB)} exemplars | {len(KEYS)} with frames | {len(pairs)} visits')
print(f'median annotation area: {C.MEDIAN_AREA:.0f} px')
print(f'{len(LABELS)} labels: {LABELS}')

## 4. Component: retrieval

Cross-scene, using each exemplar's own description as the query. Baseline is the empirical label prior, not uniform.

In [ ]:
import collections

def validate_retrieval(db, n=200, seed=0, alpha=0.5):
    rng = np.random.default_rng(seed)
    cand = [k for k, e in db.items() if (e.get('desc') or [''])[0]]
    qs = rng.choice(cand, min(n, len(cand)), replace=False)
    hits = {m: 0 for m in ('label@1','label@5','family@1','family@5')}
    tot = 0
    for k in qs:
        e = db[k]
        got = RT.retrieve(e['desc'][0], db, k=5, alpha=alpha, exclude_visit=e['visit'])
        if not got: continue
        tot += 1
        labs = [db[g]['label'] for g in got]
        fams = [C.FAMILY.get(l) for l in labs]
        tl, tf = e['label'], C.FAMILY.get(e['label'])
        hits['label@1']  += labs[0] == tl
        hits['label@5']  += tl in labs
        hits['family@1'] += fams[0] == tf
        hits['family@5'] += tf in fams
    print(f'  n={tot}  alpha={alpha}')
    for m, h in hits.items(): print(f'    {m:<10} {h/max(tot,1):6.1%}')
    pri = collections.Counter(db[k]['label'] for k in db)
    fpr = collections.Counter(C.FAMILY.get(db[k]['label']) for k in db)
    N = sum(pri.values())
    print(f'  baseline label@1 {max(pri.values())/N:.1%}  '
          f'family@1 {max(fpr.values())/N:.1%}')
    return {m: h/max(tot,1) for m, h in hits.items()}

RETRIEVAL = validate_retrieval(DB)
print('\nreference: label@5 89.0%, family@5 96.0%')

Optional: alpha sweep. `alpha=1.0` is image-only, `alpha=0.0` is description-only. 0.5 was inherited from the original notebook and never re-validated.

In [ ]:
for a in (0.0, 0.25, 0.5, 0.75, 1.0):
    validate_retrieval(DB, n=150, alpha=a)

## 5. Component: correspondence

Factorial over centering x scorer x background subtraction. Isolating one factor at a time is what caught a 30-point regression from three bundled changes.

In [ ]:
BG_SUB = 128
_F = {}

def _parts(db, k, vi, layer, center):
    ck = (k, vi, layer, center)
    if ck not in _F:
        e = db[k]; v = e.get('views', [e])[:3][vi]
        old = C.CENTER; C.CENTER = center
        f = P.patch_feats(v['crop'], layer, C.BANK_SHORT)
        C.CENTER = old
        m = P.patchify(v['mask'], *f.shape[:2])
        fg, bg = f[P._erode1(m)], f[~m]
        if len(bg) > BG_SUB:
            bg = bg[np.random.default_rng(0).choice(len(bg), BG_SUB, replace=False)]
        _F[ck] = (fg if len(fg) else None, bg if len(bg) else None)
    return _F[ck]

def _collect(db, pool, layer, center):
    FG, BG = [], []
    for k in pool:
        for vi in range(len(db[k].get('views', [db[k]])[:3])):
            fg, bg = _parts(db, k, vi, layer, center)
            if fg is not None: FG.append(fg)
            if bg is not None: BG.append(bg)
    return (np.vstack(FG) if FG else None), (np.vstack(BG) if BG else None)

def _topq(f2, V, topq=None):
    S = f2 @ V.T
    k = max(1, int(round((topq or C.TOPQ) * S.shape[1])))
    return np.sort(S, 1)[:, -k:].mean(1) if k > 1 else S.max(1)

def _proto(A):
    p = A.mean(0); return p / (np.linalg.norm(p) + 1e-8)

def factorial(db, layer=1, seeds=(0,1,2), n=120, k_ex=None):
    keys_all = list(db)
    out = {}
    for center in (False, True):
        for scorer in ('proto', 'bank'):
            for use_bg in (False, True):
                accs = []
                for seed in seeds:
                    rng = np.random.default_rng(seed)
                    sel = rng.choice(keys_all, min(n, len(keys_all)), replace=False)
                    hit = tot = 0
                    for tk in sel:
                        t = db[tk]
                        pool = [k for k, e in db.items()
                                if e['visit'] != t['visit'] and e['label'] == t['label']
                                ][:(k_ex or C.K_EX)]
                        if not pool: continue
                        FG, BG = _collect(db, pool, layer, center)
                        if FG is None: continue
                        old = C.CENTER; C.CENTER = center
                        f = P.patch_feats(t['crop'], layer, C.BANK_SHORT)
                        C.CENTER = old
                        gt = P.patchify(t['mask'], *f.shape[:2])
                        if not gt.any() or gt.all(): continue
                        f2 = f.reshape(-1, f.shape[-1])
                        if scorer == 'proto':
                            s = f2 @ _proto(FG)
                            if use_bg and BG is not None: s = s - f2 @ _proto(BG)
                        else:
                            s = _topq(f2, FG)
                            if use_bg and BG is not None: s = s - _topq(f2, BG)
                        s = s.reshape(f.shape[:2])
                        pi, pj = np.unravel_index(int(np.argmax(s)), s.shape)
                        hit += bool(gt[pi, pj]); tot += 1
                    if tot: accs.append(hit/tot)
                out[(center, scorer, use_bg)] = (float(np.mean(accs)), float(np.std(accs)))
    return out

print(f"{'center':>7} {'scorer':>7} {'bg':>5}   {'acc':>7} {'sd':>6}")
FACT = factorial(DB)
for kk in sorted(FACT, key=lambda k: -FACT[k][0]):
    c, sc, bg = kk; a, sd = FACT[kk]
    print(f'{str(c):>7} {sc:>7} {str(bg):>5}   {a:>6.1%} {sd:>6.1%}')
_F.clear()
print('\nreference: proto/bg 45.3%, bank/bg 38.9%, proto/no-bg 25.8%')

## 6. Component: box selection

The bar is the `GD top-1 (null)` baseline. Any rule that does not beat it is worse than deleting the similarity scoring.

In [ ]:
def contains(box, pt, pad=0.0):
    x0, y0, x1, y1 = box; w, h = x1-x0, y1-y0
    return (x0-pad*w <= pt[0] <= x1+pad*w) and (y0-pad*h <= pt[1] <= y1+pad*h)

def box_comparison(db, keys, n=60, seed=0):
    rng = np.random.default_rng(seed)
    sel = rng.choice(keys, min(n, len(keys)), replace=False)
    hits = {'GD rank-0 (null)': 0, 'proposed@8': 0, 'multibox k=2': 0, 'multibox k=3': 0}
    ranks, tot = [], 0
    for k in sel:
        e = db[k]
        pool = EV.pool_oracle(db, k)
        if not pool: continue
        proto = P.build_proto(pool)
        if proto is None: continue
        fp = f"{C.FRAMES}/{e['frame_file']}"
        if not os.path.exists(fp): continue
        img = cv2.cvtColor(cv2.imread(fp), cv2.COLOR_BGR2RGB)
        text = f"{(e.get('desc') or [''])[0]}. "
        text += C.PARENT_HINT.get(e['label'], C.DEFAULT_HINT)
        boxes, confs = P.gd_boxes(img, text)
        if len(boxes) == 0: continue
        gt = (np.mean(e['box'][::2]), np.mean(e['box'][1::2]))
        tot += 1
        hits['GD rank-0 (null)'] += contains(boxes[0], gt)
        hits['proposed@8'] += any(contains(b, gt) for b in boxes)
        good = [i for i, b in enumerate(boxes) if contains(b, gt)]
        if good: ranks.append(good[0])
        for kk in (2, 3):
            r = P.run(img, proto, text, topk=kk)
            if r is not None:
                hits[f'multibox k={kk}'] += contains(r['box'], gt)
    print(f'n={tot}')
    for nm, h in hits.items(): print(f'  {nm:<20} {h/max(tot,1):6.1%}')
    if ranks:
        print(f'  correct box at GD rank 0: {np.mean(np.array(ranks)==0):.1%}, '
              f'median rank {np.median(ranks):.0f}')
    return hits, tot

BOXES = box_comparison(DB, KEYS)
print('\nreference: rank-0 58.3%, best reranker 55.0%, proposed@8 96.7%')

Optional: **combined reranker, not yet adopted.** Local signal (n=40, single seed,
Apple Silicon) showed peak-only box selection degrading as `topk` widens (matches the
`multi-box top-k` sweep above), but blending GD's own detection confidence with the
correspondence peak breaks that degradation and reaches further into the 96.7%
reachable ceiling. Run this at n=80, 3 seeds (same rigor as the seed-stability check
above) before deciding whether to adopt it -- do not adopt off a single local run.

In [ ]:
def pick_box(img, proto, boxes, confs, topk, mode, alpha=0.5):
    """mode='peak': current deployed rule (score_image peak per box, take max).
    mode='combined': peak z-scored across this item's own candidate boxes,
    blended with GD's own detection confidence (also z-scored) -- comparable
    scales, no dataset-level fitting."""
    H, W = img.shape[:2]
    scored = []
    for b, cf in zip(boxes[:topk], confs[:topk]):
        x0, y0, x1, y1 = b
        pw, ph = (x1-x0)*C.BOX_PAD, (y1-y0)*C.BOX_PAD
        X0, Y0 = int(max(0, x0-pw)), int(max(0, y0-ph))
        X1, Y1 = int(min(W, x1+pw)), int(min(H, y1+ph))
        if X1-X0 < 16 or Y1-Y0 < 16: continue
        smap, _ = P.score_image(img[Y0:Y1, X0:X1], proto)
        scored.append((b, cf, float(smap.max())))
    if not scored: return None
    if mode == 'peak':
        return max(scored, key=lambda t: t[2])[0]
    peaks = np.array([s[2] for s in scored]); confs_ = np.array([s[1] for s in scored])
    pz = (peaks - peaks.mean()) / (peaks.std() + 1e-6)
    cz = (confs_ - confs_.mean()) / (confs_.std() + 1e-6)
    combo = alpha*pz + (1-alpha)*cz
    return scored[int(np.argmax(combo))][0]

def rerank_comparison(db, keys, n=80, seed=0):
    rng = np.random.default_rng(seed)
    sel = rng.choice(keys, min(n, len(keys)), replace=False)
    hits = {(topk, mode): 0 for topk in (2, 4, 8) for mode in ('peak', 'combined')}
    tot = 0
    for k in sel:
        e = db[k]
        pool = EV.pool_oracle(db, k)
        if not pool: continue
        proto = P.build_proto(pool)
        if proto is None: continue
        fp = f"{C.FRAMES}/{e['frame_file']}"
        if not os.path.exists(fp): continue
        img = cv2.cvtColor(cv2.imread(fp), cv2.COLOR_BGR2RGB)
        text = f"{(e.get('desc') or [''])[0]}. {C.PARENT_HINT.get(e['label'], C.DEFAULT_HINT)}"
        boxes, confs = P.gd_boxes(img, text)
        if len(boxes) == 0: continue
        gt = (np.mean(e['box'][::2]), np.mean(e['box'][1::2]))
        tot += 1
        for topk in (2, 4, 8):
            for mode in ('peak', 'combined'):
                picked = pick_box(img, proto, boxes, confs, topk, mode)
                if picked is not None and contains(picked, gt):
                    hits[(topk, mode)] += 1
    print(f'n={tot}')
    for topk in (2, 4, 8):
        for mode in ('peak', 'combined'):
            h = hits[(topk, mode)]
            print(f'  topk={topk}  mode={mode:<9}  {h}/{tot} = {h/max(tot,1):.1%}')
    return hits, tot

for seed in (0, 1, 2):
    print(f'--- seed {seed} ---')
    rerank_comparison(DB, KEYS, n=80, seed=seed)

### SAM point vs point+box hint -- GPU validation (was MPS n=30 only)

Reuses `EV.evaluate`/`EV.summarise` -- the same harness as the four end-to-end
conditions below -- so this is apples-to-apples with the deployed pipeline
number, not a bespoke metric. Toggles `C.SAM_BOX_PROMPT` and reruns the full
`oracle_label=True, oracle_hint=True` condition at n=80, three seeds per side,
per the notebook header's own instruction not to trust the n=30 MPS numbers
for the paper. Restores `SAM_BOX_PROMPT=True` (the notebook's stated default)
when done so later cells aren't affected by leftover global state.

In [ ]:
sam_ab = {}
for flag in (False, True):
    C.SAM_BOX_PROMPT = flag
    ious = []
    for seed in (0, 1, 2):
        R = EV.evaluate(DB, KEYS, n=80, seed=seed, oracle_label=True, oracle_hint=True)
        s = EV.summarise(R, f'SAM_BOX_PROMPT={flag} seed={seed}')
        ious.append(s['iou'])
    sam_ab[flag] = ious
    print(f'SAM_BOX_PROMPT={flag}: mean {np.mean(ious):.3f}  '
          f'per-seed {np.round(ious,3)}  spread {np.ptp(ious):.3f}')
C.SAM_BOX_PROMPT = False # restore deployed default after the A/B (off is deployed)

## 7. End-to-end 2D evaluation

Four conditions. The `retrieval in the loop` row is the honest one: no ground-truth label anywhere in the prediction path.

In [ ]:
RESULTS = {}

print('--- full pipeline (oracle label + hint) ---')
R_full = EV.evaluate(DB, KEYS, n=80, oracle_label=True, oracle_hint=True)
RESULTS['full (oracle label)'] = EV.summarise(R_full, 'full')

print('\n--- ceiling: GT box + 40% pad ---')
R_ceil = EV.evaluate(DB, KEYS, n=80, oracle_box=True)
RESULTS['ceiling (GT box)'] = EV.summarise(R_ceil, 'ceiling')

print('\n--- generic hint (no oracle affordance category) ---')
R_gen = EV.evaluate(DB, KEYS, n=80, oracle_label=True, oracle_hint=False)
RESULTS['generic hint'] = EV.summarise(R_gen, 'generic hint')

print('\n--- retrieval in the loop (no GT label at all) ---')
R_ret = EV.evaluate(DB, KEYS, n=80, oracle_label=False, oracle_hint=True)
RESULTS['retrieval in loop'] = EV.summarise(R_ret, 'retrieval')

In [ ]:
EV.distribution(R_full, 'full pipeline')
EV.distribution(R_ceil, 'ceiling')

### Seed stability

A single seed at n=80 is not enough to separate configurations a few points apart. This has already caused one false conclusion (the layer sweep).

In [ ]:
for topk in (1, 2, 3):
    ious = []
    for seed in (0, 1, 2):
        R = EV.evaluate(DB, KEYS, n=80, seed=seed, topk=topk)
        ious.append(np.mean([r['iou'] for r in R]))
    print(f'topk={topk}: mean {np.mean(ious):.3f}  per-seed {np.round(ious,3)}  '
          f'spread {np.ptp(ious):.3f}')

### Per-label breakdown

In [ ]:
import collections
by = collections.defaultdict(list)
for r in R_full: by[r['label']].append(r)
print(f"{'label':<12} {'n':>4} {'IoU':>7} {'cov':>7} {'in-GT':>7} {'zeros':>7}")
for L in sorted(by):
    v = by[L]
    print(f'{L:<12} {len(v):>4} '
          f"{np.mean([r['iou'] for r in v]):>7.3f} "
          f"{np.mean([r['coverage'] for r in v]):>7.3f} "
          f"{np.mean([r['in_gt'] for r in v]):>6.0%} "
          f"{np.mean([r['iou'] < 0.02 for r in v]):>6.0%}")

## 8. Results summary

In [ ]:
print('COMPONENT')
print(f"  retrieval label@5      {RETRIEVAL.get('label@5', float('nan')):.1%}")
print(f"  retrieval family@5     {RETRIEVAL.get('family@5', float('nan')):.1%}")
best_fact = max(FACT, key=lambda k: FACT[k][0])
print(f'  correspondence         {FACT[best_fact][0]:.1%}  {best_fact}')
print(f"  box rank-0             {BOXES[0]['GD rank-0 (null)']/max(BOXES[1],1):.1%}")
print()
print('END-TO-END (2D pixel IoU against morphologically closed GT)')
print(f"{'condition':<24} {'n':>4} {'IoU':>7} {'cov':>7} {'in-GT':>7}")
for nm, s in RESULTS.items():
    if s:
        print(f"{nm:<24} {s['n']:>4} {s['iou']:>7.3f} {s['coverage']:>7.3f} "
              f"{s['in_gt']:>6.0%}")
print()
print('CAVEATS TO STATE IN ANY WRITE-UP')
print('  - 2D pixel IoU, NOT the 3D point AP/mIoU the benchmark reports.')
print('  - Scenes selected by extraction success, not at random.')
print('  - GT fills a median 6.1% of its own bounding box (handoff 5.1).')
print('  - The oracle-label rows are upper bounds; use the retrieval row.')

## 9. Qualitative inspection (Gradio)

**Launch at most one demo per session.** Three servers holding image state is what produced 4.7 GiB of leaked GPU memory and the OOM in the earlier notebook. Restart the runtime before running quantitative evaluation afterwards.

In [ ]:
import gradio as gr
import sys
import importlib

for m in list(sys.modules):
    if m.startswith('sf3d'):
        del sys.modules[m]
from sf3d import config as C, pipeline as P, retrieval as RT, evalutils as EV

def overlay(img, res, gt=None):
    o = img.copy()
    if res is None: return o
    s = res['smap']; s = (s - s.min()) / (np.ptp(s) + 1e-8)
    hm = cv2.applyColorMap((s*255).astype(np.uint8), cv2.COLORMAP_JET)
    o = cv2.addWeighted(o, .65, cv2.cvtColor(hm, cv2.COLOR_BGR2RGB), .35, 0)
    if gt is not None and gt.any():
        e = cv2.dilate(gt.astype(np.uint8), np.ones((5,5), np.uint8)).astype(bool)
        o[e] = (o[e]*.5 + np.array([255,0,0])*.5).astype(np.uint8)
    m = res['mask']
    o[m] = (o[m]*.4 + np.array([0,255,0])*.6).astype(np.uint8)
    x0, y0, x1, y1 = [int(v) for v in res['box']]
    cv2.rectangle(o, (x0,y0), (x1,y1), (255,255,0), 3)
    cv2.circle(o, (int(res['point'][0]), int(res['point'][1])), 10, (0,0,255), -1)
    return o

def go(key, query, use_retrieval, k_ex, topk):
    e = DB[key]
    if use_retrieval:
        pool, guess = EV.pool_retrieved(DB, key, k_ex=int(k_ex))
    else:
        pool, guess = EV.pool_oracle(DB, key, k_ex=int(k_ex)), e['label']
    if not pool: return None, 'no exemplars retrieved'
    proto = P.build_proto(pool)
    img = cv2.cvtColor(cv2.imread(f"{C.FRAMES}/{e['frame_file']}"), cv2.COLOR_BGR2RGB)
    q = query.strip() or (e.get('desc') or [''])[0]
    text = f"{q}. {C.PARENT_HINT.get(guess, C.DEFAULT_HINT)}"
    res = P.run(img, proto, text, topk=int(topk))
    gt = EV.gt_full(e, *img.shape[:2])
    if res is None: return overlay(img, None), 'no boxes proposed'
    gc = EV.closed_gt(gt)
    iou = (res['mask'] & gc).sum() / max((res['mask'] | gc).sum(), 1)
    cov = (res['mask'] & gt).sum() / max(gt.sum(), 1)
    info = (f'query: {q}\n'
            f'true label: {e["label"]}   guessed: {guess}   '
            f'{"MATCH" if guess == e["label"] else "MISMATCH"}\n'
            f'prompt: {text}\n'
            f'GD conf: {res["conf"]:.2f}   peak sim: {res["peak"]:.3f}\n'
            f'mask: {res["mask"].sum()} px   (median annotation {C.MEDIAN_AREA:.0f} px)\n'
            f'IoU(closed): {iou:.3f}   coverage: {cov:.3f}\n'
            f'exemplars: ' + ', '.join(p["label"] for p in pool))
    return overlay(img, res, gt), info

with gr.Blocks(title='Stage 2 inspector') as demo:
    gr.Markdown('### Stage 2 inspector'
                '\n\nred = ground truth, green = predicted mask, '
                'yellow = selected box, blue dot = similarity peak')
    with gr.Row():
        k = gr.Dropdown(KEYS[:500], value=KEYS[0], label='exemplar', scale=3)
        q = gr.Textbox(label='query (blank = dataset description)', scale=3)
    with gr.Row():
        ur = gr.Checkbox(value=True, label='retrieval in the loop (no GT label)')
        kx = gr.Slider(1, 10, 5, step=1, label='k_ex')
        tk = gr.Slider(1, 5, C.MULTIBOX_K, step=1, label='multibox top-k')
    btn = gr.Button('run', variant='primary')
    with gr.Row():
        out = gr.Image(label='result', scale=2)
        txt = gr.Textbox(label='details', lines=10, scale=1)
    btn.click(go, [k, q, ur, kx, tk], [out, txt])

demo.launch(share=True, debug=True)

In [ ]:
RT.build_index(DB)
print('index loaded:', len(RT._R.get('keys', [])), 'exemplars')

In [ ]:
%%writefile -a /content/sf3d/retrieval.py

_raw_retrieve = retrieve

def retrieve(query_text, db, k=5, alpha=0.5, max_per_scene=2, exclude_visit=None):
    """Auto-loads the cached index if module state was cleared by a reload."""
    if 'keys' not in _R:
        build_index(db)
    return _raw_retrieve(query_text, db, k=k, alpha=alpha,
                         max_per_scene=max_per_scene, exclude_visit=exclude_visit)

In [ ]:
import sys, importlib
for m in [k for k in list(sys.modules) if k.startswith('sf3d')]:
    del sys.modules[m]
from sf3d import config as C, pipeline as P, retrieval as RT, evalutils as EV
RT.build_index(DB)
sync()
print('ready')

In [ ]:
%%writefile -a /content/sf3d/pipeline.py

# torch.set_grad_enabled() is THREAD-LOCAL. Gradio dispatches handlers to a
# worker thread, so the global setting from the notebook does not apply there.
# Decorating the entry points makes it thread-independent.
import torch as _t
patch_feats = _t.no_grad()(patch_feats)
build_proto = _t.no_grad()(build_proto)
score_image = _t.no_grad()(score_image)
gd_boxes    = _t.no_grad()(gd_boxes)
sam_mask    = _t.no_grad()(sam_mask)
run         = _t.no_grad()(run)

In [ ]:
import sys, importlib
for m in [k for k in list(sys.modules) if k.startswith('sf3d')]:
    del sys.modules[m]
from sf3d import config as C, pipeline as P, retrieval as RT, evalutils as EV
sync()
print('reloaded with no_grad decorators')

### Failure gallery

The distribution is bimodal, so the mean describes almost nobody. Look at the tails directly.

In [ ]:
import matplotlib.pyplot as plt

def gallery(R, db, worst=True, n=6):
    S = sorted(R, key=lambda r: r['iou'], reverse=not worst)[:n]
    fig, ax = plt.subplots(2, 3, figsize=(16, 10))
    for a, r in zip(ax.ravel(), S):
        e = db[r['key']]
        img = cv2.cvtColor(cv2.imread(f"{C.FRAMES}/{e['frame_file']}"), cv2.COLOR_BGR2RGB)
        pool = EV.pool_oracle(db, r['key'])
        proto = P.build_proto(pool)
        text = f"{(e.get('desc') or [''])[0]}. "
        text += C.PARENT_HINT.get(e['label'], C.DEFAULT_HINT)
        res = P.run(img, proto, text)
        gt = EV.gt_full(e, *img.shape[:2])
        a.imshow(overlay(img, res, gt)); a.axis('off')
        a.set_title(f"GT: {e['label']} | Pred: {r.get('guess', e['label'])}\nIoU {r['iou']:.3f}  cov {r['coverage']:.2f}",
                    fontsize=10)

    title = 'Worst cases' if worst else 'Best cases'
    title += '\n(Red: Ground Truth mask, Green: Predicted mask, Yellow: Selected box, Blue dot: Similarity peak)'
    plt.suptitle(title, fontsize=14)
    plt.tight_layout(); plt.show()

gallery(R_full, DB, worst=True)
gallery(R_full, DB, worst=False)